# Módulo X — Agentes Inteligentes

> Notebook companheiro — código do livro *Ciência de Dados com Python* (Howard Roatti). Licença MIT.


# Módulo X -- Agentes Inteligentes

## VISÃO GERAL

Até aqui, nossas LLMs foram **reativas**: recebem um *prompt*, produzem uma resposta e encerram. Por mais úteis que sejam, elas não *fazem* nada no mundo -- não consultam um banco de dados atualizado, não executam um cálculo confiável, não enviam um e-mail, não iteram sobre um problema até resolvê-lo. Neste módulo damos o salto da LLM que **responde** para o sistema que **age**: o **agente inteligente**.

Um agente, no sentido que exploraremos, é um sistema em que uma LLM funciona como o "motor de raciocínio" que, em um **laço** (*loop*), decide **o que fazer em seguida**, usa **ferramentas** para agir sobre o ambiente, observa os resultados e repete até cumprir um objetivo. É a ideia que está por trás dos assistentes de programação, dos automatizadores de tarefas e dos "copilotos" que se tornaram comuns.

Seguindo a linha do módulo anterior, **construiremos um agente do zero, com código Python e modelos locais** (Ollama/`transformers`), para que você entenda cada engrenagem antes de conhecer os *frameworks* que as encapsulam. Também apresentaremos, de forma conceitual e neutra, o ecossistema atual: LangChain/LangGraph, LlamaIndex e o protocolo **MCP**.

Ao final deste módulo você será capaz de:

- Definir agente à luz da IA clássica (ciclo **perceber → raciocinar → agir**) e explicar como uma LLM assume o papel de "cérebro";
- Descrever a anatomia de um agente LLM: **modelo + ferramentas + memória + orquestração**;
- Entender e implementar **uso de ferramentas** (*tool use / function calling*) e o padrão **ReAct**;
- Distinguir **memória** de curto e de longo prazo e conectá-la ao RAG (Módulo IX);
- Situar-se entre os principais *frameworks* e o protocolo **MCP**;
- Reconhecer arquiteturas **multiagente** e os **riscos** específicos de sistemas que agem de forma autônoma.

## O Que É um Agente?

O conceito de agente é **anterior** às LLMs -- ele está no coração da Inteligência Artificial clássica. Na definição consagrada por Russell e Norvig, um **agente** é qualquer entidade que **percebe** seu ambiente por meio de sensores e **age** sobre ele por meio de atuadores, buscando alcançar um objetivo. Um agente **racional** é aquele que escolhe as ações que maximizam sua expectativa de sucesso, dado o que percebeu.

Esse é exatamente o ciclo **perceber → raciocinar → agir** (*sense–think–act*), que aparece desde a robótica até os termostatos. O que mudou com as LLMs foi a peça do **"raciocinar"**: antes, a lógica de decisão precisava ser programada à mão (regras, buscas, planejadores); agora, uma LLM pode servir como um motor de raciocínio **de propósito geral**, capaz de interpretar objetivos em linguagem natural e decidir, a cada passo, qual ação tomar.

Repare na diferença em relação ao Módulo IX. Lá, a LLM era consultada e respondia. Aqui, ela é colocada **dentro de um laço** e recebe autonomia para: analisar o objetivo, decidir se precisa de uma ferramenta, escolher qual, examinar o resultado e decidir o próximo passo -- até concluir. A LLM deixa de ser um "oráculo" e passa a ser um "agente de decisão".


## Anatomia de um Agente LLM

Todo agente baseado em LLM, por mais sofisticado que seja, é composto por quatro elementos. Vale memorizá-los, pois todo *framework* apenas os organiza de formas diferentes:

1. **O Modelo (o cérebro).** A LLM que raciocina e decide. Como escolhe *ações* de forma que um programa possa executar, ela precisa produzir **saída estruturada** -- exatamente a habilidade que exercitamos no Módulo IX.
2. **As Ferramentas (as mãos).** Funções que o agente pode invocar para *agir*: uma calculadora, uma busca na web, uma consulta SQL, uma chamada de API, a recuperação de um RAG. Cada ferramenta tem um **nome**, uma **descrição** (que o modelo lê para decidir quando usá-la) e **parâmetros**.
3. **A Memória.** O que o agente "lembra": o histórico da conversa e das ações desta sessão (curto prazo) e conhecimento persistente entre sessões (longo prazo).
4. **A Orquestração (o laço).** O código que costura tudo: envia o estado ao modelo, recebe a decisão, executa a ferramenta escolhida, devolve o resultado ao modelo e repete, até uma condição de parada.

## Uso de Ferramentas (*Tool Use / Function Calling*)

Esta é a capacidade que transforma uma LLM em agente. A ideia é engenhosamente simples: **descrevemos** ao modelo quais ferramentas existem e o instruímos a, quando quiser usar uma, **responder com uma estrutura** (tipicamente um JSON) dizendo *qual* ferramenta e com *quais argumentos* -- em vez de responder ao usuário. O nosso código detecta essa estrutura, **executa de fato** a função correspondente e devolve o resultado ao modelo, que então continua.

É crucial entender o que a LLM **faz e não faz**: ela **não executa** a ferramenta. Ela apenas **decide** e **preenche os argumentos**. Quem executa é o nosso programa. A LLM é o cérebro que dá a ordem; as mãos são o código Python. Essa separação é também uma fronteira de segurança (Módulo XI).

Considere uma ferramenta de clima. Nós a descrevemos assim (formato aberto, compatível com Ollama e com o padrão de *function calling* usado pelas principais APIs):


In [ ]:
ferramentas = [{
    "type": "function",
    "function": {
        "name": "consultar_clima",
        "description": "Retorna a temperatura atual de uma cidade.",
        "parameters": {
            "type": "object",
            "properties": {
                "cidade": {"type": "string", "description": "Nome da cidade"}
            },
            "required": ["cidade"],
        },
    },
}]

Quando perguntamos "Qual a temperatura em Recife?", o modelo -- em vez de inventar um número (alucinação!) -- responde pedindo a chamada `consultar_clima(cidade="Recife")`. Nosso código executa a função real (que pode consultar uma API meteorológica) e devolve o valor; o modelo então redige a resposta final ao usuário com o dado **correto**. Note como isso ataca diretamente duas limitações do Módulo VIII: o **corte de conhecimento** (a ferramenta traz dados atuais) e a **alucinação** (o número vem de uma fonte real, não da imaginação do modelo).


In [ ]:
import ollama

def consultar_clima(cidade: str) -> str:
    # numa aplicação real, aqui iria uma chamada a uma API meteorológica
    return f"A temperatura em {cidade} é 28°C."

mensagens = [{"role": "user", "content": "Qual a temperatura em Recife?"}]
resposta = ollama.chat(model="llama3.2:3b", messages=mensagens, tools=ferramentas)

# o modelo decidiu chamar uma ferramenta?
for chamada in resposta["message"].get("tool_calls", []):
    nome = chamada["function"]["name"]
    args = chamada["function"]["arguments"]
    if nome == "consultar_clima":
        resultado = consultar_clima(**args)                       # NOSSO código executa
        mensagens.append(resposta["message"])                     # registra a decisão do modelo
        mensagens.append({"role": "tool", "content": resultado})  # devolve a observação

# segunda passada: agora o modelo redige a resposta final com o dado real
final = ollama.chat(model="llama3.2:3b", messages=mensagens, tools=ferramentas)
print(final["message"]["content"])

## O Padrão ReAct: Raciocinar + Agir

O uso de ferramentas em uma única ida e volta já é poderoso, mas o verdadeiro comportamento agêntico surge quando **iteramos**. O padrão **ReAct** (Yao et al., 2022) -- de *Reasoning + Acting* -- estrutura o laço do agente em uma sequência de três momentos que se repetem:

- **Pensamento** (*Thought*): o modelo raciocina, em linguagem natural, sobre o estado atual e o que fazer ("preciso primeiro descobrir X, então vou usar a ferramenta Y").
- **Ação** (*Action*): o modelo escolhe uma ferramenta e seus argumentos.
- **Observação** (*Observation*): o resultado da ferramenta, devolvido ao modelo.

O ciclo `Pensamento → Ação → Observação` repete-se, com cada observação alimentando o próximo pensamento, até que o modelo julgue ter informação suficiente e produza a **resposta final**. É a materialização do *chain-of-thought* do Módulo IX, agora **intercalado com ações no mundo**.

O laço de orquestração ReAct, implementado à mão, deixa a mecânica transparente:


In [ ]:
import ollama

# --- ferramentas do agente ---
def calculadora(expressao: str) -> str:
    try:
        return str(eval(expressao, {"__builtins__": {}}))  # sandbox mínimo (ver Módulo XI)
    except Exception as e:
        return f"erro: {e}"

def buscar_base(consulta: str) -> str:
    # aqui entraria um RAG (Módulo IX); simplificado para o exemplo
    fatos = {"capital do brasil": "Brasília", "populacao de brasilia": "cerca de 2,8 milhões"}
    return fatos.get(consulta.lower(), "sem resultado")

TOOLS = {"calculadora": calculadora, "buscar_base": buscar_base}

esquema = [
    {"type": "function", "function": {"name": "calculadora",
        "description": "Avalia uma expressão matemática. Ex: '2*(3+4)'.",
        "parameters": {"type": "object",
            "properties": {"expressao": {"type": "string"}}, "required": ["expressao"]}}},
    {"type": "function", "function": {"name": "buscar_base",
        "description": "Busca um fato numa base de conhecimento interna.",
        "parameters": {"type": "object",
            "properties": {"consulta": {"type": "string"}}, "required": ["consulta"]}}},
]

def agente(objetivo: str, max_passos: int = 5) -> str:
    msgs = [
        {"role": "system", "content": "Você é um agente. Use ferramentas quando precisar. "
                                      "Quando tiver a resposta final, responda diretamente."},
        {"role": "user", "content": objetivo},
    ]
    for passo in range(max_passos):                 # laço = orquestração; limite evita loop infinito
        r = ollama.chat(model="llama3.2:3b", messages=msgs, tools=esquema)
        chamadas = r["message"].get("tool_calls")
        if not chamadas:                            # sem ação => resposta final (condição de parada)
            return r["message"]["content"]
        msgs.append(r["message"])
        for c in chamadas:                          # AÇÃO + OBSERVAÇÃO
            nome, args = c["function"]["name"], c["function"]["arguments"]
            observacao = TOOLS[nome](**args)
            print(f"[passo {passo}] ação={nome}{args} -> obs={observacao}")
            msgs.append({"role": "tool", "content": observacao})
    return "Limite de passos atingido."

print(agente("Quanto é a população de Brasília multiplicada por 2? Responda o número final."))

Esse pequeno programa exibe **todos** os elementos da anatomia: o modelo (cérebro), as ferramentas (mãos), a lista de mensagens (memória de curto prazo) e o laço `for` (orquestração), com uma **condição de parada** e um **limite de passos** -- salvaguarda essencial, pois um agente sem limite pode entrar em laço e consumir recursos indefinidamente (tema do Módulo XI).

## Memória

Um agente sem memória é como um profissional com amnésia a cada minuto. Distinguimos dois tipos:

- **Memória de curto prazo (contexto de trabalho):** o histórico da sessão atual -- as mensagens, os pensamentos, as ações e observações. Ela vive na **janela de contexto** do modelo (Módulo VIII). É por isso que a janela de contexto importa tanto para agentes: um laço ReAct longo acumula muitas mensagens, e há um limite. Estratégias para lidar com isso incluem **resumir** o histórico antigo e manter apenas o essencial.
- **Memória de longo prazo (persistente):** conhecimento que sobrevive entre sessões -- preferências do usuário, fatos aprendidos, documentos. A implementação canônica é... **RAG** (Módulo IX): guardamos "memórias" como *embeddings* em um banco vetorial e **recuperamos** as relevantes quando necessário, injetando-as no contexto. Ou seja, o RAG que aprendemos como técnica de conhecimento reaparece aqui como **memória de longo prazo do agente**. Conceitos do curso se encadeando.

## Planejamento e Decomposição de Tarefas

Para objetivos complexos, agentes eficazes não improvisam passo a passo: eles **planejam**. Uma estratégia comum é pedir ao modelo que primeiro **decomponha** a meta em subtarefas ("para escrever este relatório, preciso: 1. coletar os dados; 2. analisá-los; 3. gerar os gráficos; 4. redigir"), e então executá-las em ordem, eventualmente revisando o plano à luz das observações. Padrões mais avançados (como *Plan-and-Execute* e *Tree of Thoughts*) exploram e comparam múltiplos caminhos antes de agir -- uma ideia que ecoa as buscas em árvore da IA clássica, agora guiadas por uma LLM.

## *Frameworks* e o Protocolo MCP

Construímos nosso agente à mão para entender a mecânica. Em produção, *frameworks* poupam esse trabalho repetitivo (gerência do laço, memória, integração de ferramentas, tolerância a falhas). Um panorama **neutro**:

- **LangChain / LangGraph:** o ecossistema mais difundido. O **LangGraph** modela o agente como um **grafo de estados**, dando controle fino sobre o fluxo -- útil quando o laço precisa de ramificações, ciclos e pontos de revisão humana.
- **LlamaIndex:** originalmente focado em ingestão/indexação de dados para RAG (Módulo IX), hoje também oferece abstrações de agentes -- forte quando o coração da aplicação é conhecimento sobre documentos.
- Há ainda opções voltadas a **multiagentes** (por exemplo, CrewAI, AutoGen), que facilitam orquestrar vários agentes com papéis distintos.

Um desenvolvimento importante e transversal a todos eles é o **MCP (*Model Context Protocol*)**, um **protocolo aberto** (proposto pela Anthropic e adotado amplamente pela indústria) que padroniza *como* um modelo se conecta a ferramentas e fontes de dados externas. A analogia usual é a de uma **"porta USB-C para IA"**: em vez de cada aplicação implementar integrações sob medida para cada ferramenta, um **servidor MCP** expõe ferramentas e dados de forma padronizada, e qualquer **cliente MCP** (uma aplicação de IA) pode consumi-los. Isso torna as ferramentas **reutilizáveis e interoperáveis** entre diferentes modelos e aplicações -- um passo de amadurecimento do ecossistema, comparável à padronização que APIs REST trouxeram para a web.

> **Nota didática.** *Frameworks* mudam rápido; os **conceitos** deste módulo (ciclo perceber-raciocinar-agir, tool use, ReAct, memória, orquestração) são estáveis e transferem-se para qualquer ferramenta. Aprenda os conceitos; troque de *framework* sem medo.

## Sistemas Multiagente

Assim como equipes humanas dividem trabalho, às vezes é vantajoso ter **vários agentes especializados** cooperando, em vez de um único agente "faz-tudo". Padrões comuns:

- **Supervisor e trabalhadores:** um agente **orquestrador** decompõe a tarefa e delega a agentes especialistas (um para pesquisa, um para escrita, um para revisão), integrando os resultados.
- **Papéis colaborativos:** agentes com personas distintas (ex.: "programador" e "revisor") interagem, criticando e melhorando o trabalho uns dos outros -- um eco da dinâmica adversarial das GANs (Módulo VII), agora no nível de raciocínio.

Multiagentes podem melhorar qualidade e modularidade, mas aumentam **custo, latência e complexidade** de depuração. A recomendação prática é começar simples (um agente, poucas ferramentas) e só evoluir para multiagentes quando houver ganho claro.


## Projeto Guiado: Um Agente de Pesquisa com RAG

Como fecho prático, o projeto sugerido combina **tudo** dos Módulos IX e X: um agente que responde perguntas sobre uma base de documentos e faz contas quando necessário.

- **Ferramenta 1 -- `buscar_documentos(consulta)`:** um RAG completo (Módulo IX) sobre os seus PDFs/notas -- é a **memória de longo prazo** do agente.
- **Ferramenta 2 -- `calculadora(expressao)`:** para números confiáveis.
- **Orquestração:** o laço ReAct deste módulo, com limite de passos.
- **Comportamento esperado:** diante de "Segundo o manual, qual o desconto máximo, e quanto seria em um pedido de R\$ 4.000?", o agente **primeiro** recupera a regra de desconto (ferramenta 1), **depois** calcula o valor (ferramenta 2) e **então** redige a resposta final -- citando a fonte e sem inventar números.

O roteiro detalhado de implementação está entre os exercícios; ele integra o RAG do Módulo IX com o laço de agente deste módulo.

## Dos Poderes aos Riscos

Um agente que age no mundo -- executa código, acessa a internet, escreve em bancos de dados, gasta dinheiro por token -- é muito mais poderoso e, por isso mesmo, muito mais **arriscado** que uma LLM que apenas conversa. Note no nosso código o `eval` "com `__builtins__` vazio" e o `max_passos`: são embriões de **salvaguardas**. Um agente real levanta questões sérias: e se ele executar uma ação destrutiva? E se um documento malicioso, recuperado pelo RAG, contiver instruções escondidas que "sequestram" o agente (*prompt injection*)? Como limitar o que ele pode fazer? Como auditar suas decisões e custos?

Essas perguntas -- segurança, permissões, *guardrails*, avaliação, ética e a operação em produção (LLMOps) -- são o objeto do **Módulo XI -- IA Responsável, Avaliação e Produção**, o fechamento do curso.

## Síntese do Módulo

Demos o salto da LLM reativa para o **agente** que percebe, raciocina e age. Ancoramos o conceito na IA clássica (agente racional; ciclo *sense–think–act*) e mostramos a LLM assumindo o papel de motor de raciocínio. Dissecamos a anatomia -- **modelo, ferramentas, memória, orquestração** -- e implementamos, do zero e localmente, **uso de ferramentas** e o padrão **ReAct**, entendendo que a LLM *decide* mas é o nosso código que *executa*. Conectamos a **memória de longo prazo** ao RAG do Módulo IX, discutimos **planejamento**, situamo-nos entre os *frameworks* (LangGraph, LlamaIndex) e o protocolo **MCP**, e vimos quando arquiteturas **multiagente** valem a pena. Por fim, expusemos os **riscos** que motivam o próximo e último módulo.

## Exercícios

1. **Conceitual.** Diferencie, com um exemplo próprio, uma LLM "reativa" (Módulo IX) de um "agente" (Módulo X). Qual elemento da anatomia é o que efetivamente cria o comportamento autônomo?
2. **Tool use.** Rode o exemplo de `consultar_clima`. Em seguida, adicione uma segunda ferramenta (`converter_moeda`) e faça uma pergunta que exija que o modelo escolha *qual* ferramenta usar. O que na definição da ferramenta orienta essa escolha?
3. **ReAct (implementação).** Execute o agente ReAct do módulo e observe o *log* de passos. Depois, formule uma pergunta que exija **duas** ferramentas em sequência (uma busca seguida de um cálculo) e verifique se o agente encadeia as ações corretamente.
4. **Salvaguardas.** Explique por que o `max_passos` e o `eval` com `__builtins__` vazio são importantes. Proponha uma salvaguarda adicional para a ferramenta `calculadora` que impeça expressões perigosas.
5. **Memória de longo prazo.** Descreva, em alto nível, como você transformaria o histórico de conversas de um usuário em uma "memória de longo prazo" usando *embeddings* e um banco vetorial (Módulo IX). Quando o agente deveria recuperar essas memórias?
6. **MCP (pesquisa).** Explique, com suas palavras e a analogia do "USB-C para IA", qual problema o protocolo MCP resolve. Cite uma vantagem prática de ferramentas padronizadas por MCP frente a integrações sob medida.
7. **Multiagente.** Projete (apenas o desenho, com papéis e fluxo) um sistema de dois ou três agentes para produzir um relatório de análise de dados a partir de um CSV, reutilizando conceitos dos Módulos IV, V e VI. Justifique a divisão de papéis.
8. **Projeto integrador.** Implemente o "Agente de Pesquisa com RAG" descrito no módulo: acople a ferramenta de RAG do Módulo IX ao laço ReAct deste módulo e à `calculadora`. Teste com uma pergunta que exija recuperar uma informação **e** fazer um cálculo sobre ela.
